# 06 Official experiments (designs A, B, C)

This notebook shows and re-runs the **official** experiments (2010–2023; 2024 is locked).

| Design | What it does |
|---|---|
| **A (primary)** | Expanding window: train on 2010…T−1, rebuild year T, for T = 2012…2023 |
| **B (learning curve)** | Test years 2019, 2021, 2023; train on the k years just before (k = 2…) |
| **C (secondary)** | Leave-one-year-out, as in the pilot |

Models: B0 uniform, B1 seasonal profile, B4 base-paper DM, B4s + annual size, **M1** + local climate + El Niño (ONI), M1-noONI, M1-short, M1-shufTT / M1-shufTest (checks), M2 boosted trees.

**How to use:**
1. Run the setup and data cells. Run notebook 00 first if the dengue/NASA POWER data are missing.
2. Section 2 shows the committed official results; nothing is recomputed.
3. Section 3 re-runs a design yourself. Start with `SUBSET = 60` (about 3–6 min per design on Colab CPU). `SUBSET = 300` reproduces the official numbers (about 25–55 min per design). A GPU is not needed.

In [ ]:
# ---- Setup: works in Google Colab and locally (VS Code / Jupyter) ----
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/aminah9945/dengue-climate-reconstruction"
BRANCH = "main"          # use "official-run" if the pull request is not merged yet
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/dengue-climate-reconstruction")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[all]"], check=True)
    # Optional: keep the downloaded data on Google Drive so it survives the session.
    USE_DRIVE = False
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ.setdefault("DATA_DIR", "/content/drive/MyDrive/dengue_recon_data")
else:
    # Local: this notebook lives in <repo>/notebooks/
    REPO = Path(os.environ.get("DENGUE_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())).resolve()

os.environ.setdefault("DENGUE_ROOT", str(REPO))
os.environ.setdefault("DATA_DIR", str(REPO / "data"))
os.environ.setdefault("RESULTS_DIR", str(REPO / "results"))
os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=1")
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))      # makes the package importable without a kernel restart

from dengue_recon import config as C
C.ensure_dirs()
print("repo       :", C.ROOT)
print("DATA_DIR   :", C.DATA_DIR, "| processed data present:", (C.DATA_DIR / "dengue_weekly.parquet").exists())
print("RESULTS_DIR:", C.RESULTS_DIR)

In [ ]:
import json
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)


def show(path, width=900):
    p = Path(path)
    p = p if p.is_absolute() else C.FIG_DIR / p
    display(Image(str(p), width=width)) if p.exists() else print("missing figure:", p)


def have_data():
    ok = (C.DATA_DIR / "dengue_weekly.parquet").exists()
    if not ok:
        print("Processed data not found in DATA_DIR - run notebook 00 first (or set DATA_DIR).")
    return ok
OFF = C.ROOT / 'results' / 'official'
TAB = OFF / 'tables'
FIG = OFF / 'figures'


## 1. Extra inputs: El Niño index (ONI) and, optionally, CHIRPS rainfall

In [ ]:
from dengue_recon.data import download, oni
if not (C.DATA_DIR / "oni_weekly.parquet").exists():
    oni.main()                                  # NOAA CPC download + weekly table, a few seconds
print("ONI ready:", (C.DATA_DIR / "oni_weekly.parquet").exists())

GET_CHIRPS = False   # only needed for the CHIRPS robustness run; ~1-2 h for 300 municipalities
if GET_CHIRPS and not (C.DATA_DIR / "climate_weekly_chirps.parquet").exists():
    download.main(["weather", "--source", "chirps"])
    subprocess.run([sys.executable, "-m", "dengue_recon.data.build_dataset", "--source", "chirps"], check=True)
have_data()

## 2. Official results (committed, 300 municipalities)

In [ ]:
print((OFF / "OFFICIAL_RESULTS.md").read_text()[:3000])

In [ ]:
display(Markdown("**Hypothesis decisions**"))
display(pd.read_csv(TAB / "hypotheses.csv"))
display(Markdown("**Model means per run**"))
display(pd.read_csv(TAB / "model_means.csv").round(4))

In [ ]:
for f in ["figA_designA_jsd_by_test_year.png", "figB_learning_curves.png", "figC_gain_with_CIs.png",
          "figD_gain_by_region.png", "figE_shuffle_test.png", "figF_weather_source.png", "figG_climate_coefficients.png"]:
    display(Markdown(f"**{f}**")); show(FIG / f, width=1000)

## 3. Re-run a design yourself

Outputs go to a separate folder (`results/official_rerun/`), so the committed results are never overwritten. 2024 stays locked.

In [ ]:
DESIGN  = "A"        # "A", "B" or "C"
CLIMATE = "power"    # "power" or "chirps"
SUBSET  = 60         # 60 = quick check; 300 = full official run
RUN = False          # set True to run

OUT = C.ROOT / "results" / "official_rerun"
if RUN and have_data():
    cmd = [sys.executable, str(C.ROOT / "scripts" / "run_pipeline.py"), "--step", "official",
           "--design", DESIGN, "--climate", CLIMATE, "--subset", str(SUBSET), "--out", str(OUT)]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
import glob
for f in sorted(glob.glob(str(OUT / "**" / "model_summary_*.csv"), recursive=True)) + \
         sorted(glob.glob(str(OUT / "**" / "metrics_detail_*.csv"), recursive=True))[:0]:
    display(Markdown(f"**{Path(f).relative_to(OUT)}**")); display(pd.read_csv(f).round(4))
if not list(OUT.glob("**/model_summary_*.csv")) and OUT.exists():
    # summaries are written by the post-check script
    for d in OUT.iterdir():
        if d.is_dir():
            subprocess.run([sys.executable, str(C.ROOT / "scripts" / "official_postcheck.py"), str(d)], check=False)
    for f in OUT.glob("**/model_summary_*.csv"):
        display(Markdown(f"**{f.relative_to(OUT)}**")); display(pd.read_csv(f).round(4))

**Compare:** with `SUBSET = 300`, the model means should match `model_means.csv` above for the same design and climate. With `SUBSET = 60` the numbers differ (fewer municipalities); this is only a check that the code runs.